# Zomato Analytics — Bronze Layer (Raw Ingestion)

**Purpose**: Ingest raw data from source systems into the Bronze (landing) layer with
minimal transformation. Data is stored as-is to preserve full lineage and enable
reprocessing from source.

**Unity Catalog**: All objects are created under a **dedicated catalog** (`zomato_analytics`)
with schema `bronze`. This ensures **zero impact** on any existing catalogs, schemas, or tables.

**Namespace**: `zomato_analytics.bronze.<table_name>`

**Data Sources**:
- `customers` — customer registration & profile data
- `restaurants` — restaurant onboarding & metadata
- `orders` — order transactions
- `deliveries` — delivery tracking events
- `reviews` — customer reviews & ratings
- `delivery_partners` — **Lakebase (Postgres) table, full or incremental load with a watermark** (Exercise 5)

**Load type**: the five file sources are complete daily file drops, so they are always **overwritten** (full load).
The Lakebase partners are loaded **full or incremental** — chosen by one flag, `lakebase_load_type` (Exercise 5).

## Exercise 5 — Lakebase partners into Bronze: how it works

**The use case.** The delivery-partner master (name, city, vehicle, status, rating, KYC …) lives in a **Lakebase
Postgres** table that the partner app changes all day. We copy it into the lakehouse and keep the history of every
partner. Instead of copying the whole table on every run, we copy **only the rows that changed since the last run**.
To know what *"since the last run"* means we keep a **watermark**: the newest `updated_at` value already loaded.

```
LAKEBASE (Postgres)   zomato_ops.delivery_partners      one row per partner
                      updated_at = time of the row's last change  (the watermark column)
      │
      │  ① BRONZE   full        → read ALL partners
      │             incremental → read only partners with updated_at > watermark   ◀── THIS NOTEBOOK
      ▼
bronze.delivery_partners              every partner row received from Lakebase
bronze.delivery_partners_watermark    ONE row: the newest updated_at loaded so far
      │
      │  ② SILVER   newest row per partner, cleaned → overwrite
      ▼
silver.delivery_partners              one row per partner = how the partner looks NOW
      │
      │  ③ GOLD     compare with the current version → SCD Type 2
      ▼
gold.dim_delivery_partners            one row per partner per version = full history
```

### Bronze algorithm (section 6A)

```
1. Read the flag            lakebase_load_type = incremental | full
2. Decide what to read      full        → every partner
                            incremental → only partners with updated_at > watermark
                                          (no watermark yet = first run → every partner)
3. Write to Bronze          full → overwrite        incremental → append
4. Save the new watermark   = newest updated_at among the rows just read
                            (nothing read → the watermark stays as it is)
```

**Example**

| Run | Flag | Watermark before | Rows read from Lakebase | Bronze | Watermark after |
|---|---|---|---|---|---|
| 1 | incremental | *(none — first run)* | 5,001 (all) | append | 10 Nov 08:00 |
| 2 | incremental | 10 Nov 08:00 | 96 (changed after 08:00) | append | 10 Nov 09:00 |
| 3 | incremental | 10 Nov 09:00 | 0 (nothing changed) | — | 10 Nov 09:00 |
| 4 | full | *(not used)* | 5,020 (all) | overwrite | 10 Nov 10:00 |

**Good to know**
- **Safe to re-run.** If a run stops after step 3, the next run reads the same rows again — Silver keeps only the
  newest row per partner, so nothing is counted twice.
- The watermark is saved as text **exactly as Lakebase returned it** (with its time zone), so no time-zone
  conversion can shift it.
- This simple watermark relies on the app setting `updated_at` on every change (deletes are soft deletes) and on
  short transactions, like the app in `setup/01_generate_data`.

**One flag decides the load type** — `lakebase_load_type` (widget in the Bronze notebook / job parameter):

| | `incremental` (default) | `full` |
|---|---|---|
| Bronze reads from Lakebase | only partners with `updated_at` > watermark (very first run: all partners) | all partners |
| Bronze writes | **append** | **overwrite** |
| Watermark after the run | newest `updated_at` that was read | newest `updated_at` that was read |
| Silver | rebuilt from Bronze (**overwrite**) | same |
| Gold | **SCD Type 2** — a new version only for partners that changed | same — the history is kept |
| Use it for | every scheduled run | the first load, or to copy everything again after a problem |

It is the **only** load-type flag in the pipeline: the file sources (customers, orders, …) are complete daily file
drops, so Bronze always overwrites them.

### 🔐 How Bronze connects to Lakebase securely
- **No secret is stored anywhere** — `WorkspaceClient()` uses the identity running this notebook / job.
- The Postgres password is a **1-hour OAuth token** from `w.postgres.generate_database_credential(...)`, created right
  before connecting and never printed or saved.
- The connection is **encrypted** (`sslmode="require"`).
- The reading identity only needs `SELECT` on `zomato_ops.delivery_partners`.
- Full checklist (service principals, grants, secret scopes, CI/CD): see the top of `setup/01_generate_data`.

## 0. Install Dependencies (Lakebase client)

In [ ]:
%pip install -q "psycopg[binary]>=3.1" "databricks-sdk>=0.81.0"
dbutils.library.restartPython()

## 1. Configuration & Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, current_timestamp, lit, sha2, concat_ws,
)
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, BooleanType,
    IntegerType, TimestampType, ArrayType, FloatType,
)

## 2. Pipeline Parameters

In [ ]:
# Notebook widgets for parameterized runs
dbutils.widgets.text("env", "dev", "Environment (dev/staging/prod)")
dbutils.widgets.text("catalog_name", "zomato_analytics", "Unity Catalog name")
dbutils.widgets.text("source_path", "/Volumes/zomato_analytics/raw/landing", "Source data path")

ENV = dbutils.widgets.get("env")
CATALOG = dbutils.widgets.get("catalog_name")
SOURCE_PATH = dbutils.widgets.get("source_path")

# Fixed schema name — all Bronze tables live here
SCHEMA = "bronze"

# Fully qualified namespace: catalog.schema
FQN = f"{CATALOG}.{SCHEMA}"

print(f"Environment  : {ENV}")
print(f"Catalog      : {CATALOG}")
print(f"Schema       : {SCHEMA}")
print(f"Namespace    : {FQN}")
print(f"Source Path  : {SOURCE_PATH}")

In [ ]:
# --- Lakebase source (Exercise 5) ---
# lakebase_load_type is the ONE load-type flag: "incremental" (watermark) or "full". File sources are always full.
dbutils.widgets.dropdown("lakebase_load_type", "incremental", ["incremental", "full"], "Lakebase load type")
dbutils.widgets.text("lakebase_project_id", "zomato-ops", "Lakebase project id")
dbutils.widgets.text("lakebase_branch_id", "production", "Lakebase branch id")
dbutils.widgets.text("lakebase_database", "databricks_postgres", "Lakebase Postgres database")

LAKEBASE_LOAD_TYPE = dbutils.widgets.get("lakebase_load_type")
LAKEBASE_PROJECT_ID = dbutils.widgets.get("lakebase_project_id")
LAKEBASE_BRANCH_ID = dbutils.widgets.get("lakebase_branch_id")
LAKEBASE_DATABASE = dbutils.widgets.get("lakebase_database")

SOURCE_TABLE = "zomato_ops.delivery_partners"                      # in Lakebase
BRONZE_PARTNERS_TABLE = f"{FQN}.delivery_partners"                 # Bronze data
WATERMARK_TABLE = f"{FQN}.delivery_partners_watermark"             # Bronze watermark (one row)

print(f"Lakebase load: {LAKEBASE_LOAD_TYPE}  ({SOURCE_TABLE} → {BRONZE_PARTNERS_TABLE})")

## 3. Create Dedicated Catalog & Schema (Idempotent)

These statements use `IF NOT EXISTS` — they will **never** modify or drop
any existing catalog, schema, or table in your workspace.

In [ ]:
# Create dedicated catalog — isolated from all other catalogs
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"COMMENT ON CATALOG {CATALOG} IS 'Zomato Analytics DE Project — Isolated catalog for medallion architecture'")

# Create bronze schema inside our catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQN}")
spark.sql(f"COMMENT ON SCHEMA {FQN} IS 'Bronze layer — raw ingested data with audit columns'")

# Set current catalog & schema for this notebook
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"✓ Catalog '{CATALOG}' ready")
print(f"✓ Schema '{FQN}' ready")
print(f"✓ Current context: {CATALOG}.{SCHEMA}")

## 4. Schema Definitions
Explicit schemas enforce data contracts at ingestion time.

In [ ]:
SCHEMAS = {
    "customers": StructType([
        StructField("customer_id", StringType(), False),
        StructField("first_name", StringType(), True),
        StructField("last_name", StringType(), True),
        StructField("email", StringType(), True),
        StructField("phone", StringType(), True),
        StructField("city", StringType(), True),
        StructField("locality", StringType(), True),
        StructField("pincode", StringType(), True),
        StructField("latitude", DoubleType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("signup_date", StringType(), True),
        StructField("subscription_tier", StringType(), True),
        StructField("is_active", BooleanType(), True),
        StructField("preferred_payment", StringType(), True),
        StructField("preferred_cuisine", StringType(), True),
        StructField("total_orders_lifetime", IntegerType(), True),
        StructField("avg_order_value", DoubleType(), True),
        StructField("_ingested_at", StringType(), True),
        StructField("_source_system", StringType(), True),
    ]),
    "restaurants": StructType([
        StructField("restaurant_id", StringType(), False),
        StructField("name", StringType(), True),
        StructField("city", StringType(), True),
        StructField("locality", StringType(), True),
        StructField("address", StringType(), True),
        StructField("pincode", StringType(), True),
        StructField("latitude", DoubleType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("cuisines", ArrayType(StringType()), True),
        StructField("restaurant_type", StringType(), True),
        StructField("avg_cost_for_two", IntegerType(), True),
        StructField("rating", FloatType(), True),
        StructField("total_reviews", IntegerType(), True),
        StructField("is_delivering_now", BooleanType(), True),
        StructField("has_online_delivery", BooleanType(), True),
        StructField("has_table_booking", BooleanType(), True),
        StructField("is_premium_partner", BooleanType(), True),
        StructField("onboarded_date", StringType(), True),
        StructField("owner_name", StringType(), True),
        StructField("owner_phone", StringType(), True),
        StructField("gst_number", StringType(), True),
        StructField("_ingested_at", StringType(), True),
        StructField("_source_system", StringType(), True),
    ]),
    "orders": StructType([
        StructField("order_id", StringType(), False),
        StructField("customer_id", StringType(), True),
        StructField("restaurant_id", StringType(), True),
        StructField("order_placed_at", StringType(), True),
        StructField("order_status", StringType(), True),
        StructField("payment_method", StringType(), True),
        StructField("payment_status", StringType(), True),
        StructField("platform", StringType(), True),
        StructField("subtotal", DoubleType(), True),
        StructField("discount_pct", DoubleType(), True),
        StructField("discount_amount", DoubleType(), True),
        StructField("coupon_code", StringType(), True),
        StructField("delivery_fee", DoubleType(), True),
        StructField("tax_amount", DoubleType(), True),
        StructField("total_amount", DoubleType(), True),
        StructField("num_items", IntegerType(), True),
        StructField("special_instructions", StringType(), True),
        StructField("_ingested_at", StringType(), True),
        StructField("_source_system", StringType(), True),
    ]),
    "deliveries": StructType([
        StructField("delivery_id", StringType(), False),
        StructField("order_id", StringType(), True),
        StructField("delivery_partner_id", StringType(), True),
        StructField("delivery_partner_name", StringType(), True),
        StructField("partner_phone", StringType(), True),
        StructField("vehicle_type", StringType(), True),
        StructField("pickup_timestamp", StringType(), True),
        StructField("delivered_timestamp", StringType(), True),
        StructField("delivery_time_mins", IntegerType(), True),
        StructField("delivery_status", StringType(), True),
        StructField("delivery_distance_km", DoubleType(), True),
        StructField("delivery_rating", IntegerType(), True),
        StructField("_ingested_at", StringType(), True),
        StructField("_source_system", StringType(), True),
    ]),
    "reviews": StructType([
        StructField("review_id", StringType(), False),
        StructField("order_id", StringType(), True),
        StructField("customer_id", StringType(), True),
        StructField("restaurant_id", StringType(), True),
        StructField("rating", IntegerType(), True),
        StructField("review_text", StringType(), True),
        StructField("review_timestamp", StringType(), True),
        StructField("sentiment_label", StringType(), True),
        StructField("is_verified_order", BooleanType(), True),
        StructField("upvotes", IntegerType(), True),
        StructField("has_photo", BooleanType(), True),
        StructField("_ingested_at", StringType(), True),
        StructField("_source_system", StringType(), True),
    ]),
}

## 5. Bronze Ingestion Functions

In [ ]:
def ingest_to_bronze(
    table_name: str,
    source_format: str = "parquet",
) -> int:
    """
    Ingest a raw dataset into the Bronze Delta table using Unity Catalog.
    Each file drop is a complete copy of its source, so the table is always overwritten (full load).

    All tables are created under: {CATALOG}.{SCHEMA}.{table_name}

    Adds audit columns:
      - _bronze_loaded_at : ingestion timestamp
      - _source_file      : originating file path
      - _row_hash         : SHA-256 hash of the row (dedup)

    Returns the number of records written.
    """
    source_file = f"{SOURCE_PATH}/{table_name}"
    schema = SCHEMAS.get(table_name)

    # Read raw data with enforced schema
    raw_df = (
        spark.read
        .format(source_format)
        .schema(schema)
        .option("mergeSchema", "true")
        .load(source_file)
    )

    # Add audit / lineage columns
    enriched_df = (
        raw_df
        .withColumn("_bronze_loaded_at", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn(
            "_row_hash",
            sha2(concat_ws("||", *[col(c) for c in raw_df.columns]), 256),
        )
    )

    # Fully qualified table name: catalog.schema.table
    target_table = f"{FQN}.{table_name}"

    # Full load — overwrite (creates the table if it does not exist yet)
    (
        enriched_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    record_count = enriched_df.count()
    print(f"  ✓ {target_table}: {record_count:,} records ingested (full)")
    return record_count

## 6. Execute Bronze Ingestion Pipeline

In [ ]:
pipeline_tables = [
    {"table_name": "customers"},
    {"table_name": "restaurants"},
    {"table_name": "orders"},
    {"table_name": "deliveries"},
    {"table_name": "reviews"},
]

total_records = 0
pipeline_status = []

for tbl_cfg in pipeline_tables:
    try:
        count = ingest_to_bronze(**tbl_cfg)
        total_records += count
        pipeline_status.append({"table": tbl_cfg["table_name"], "status": "SUCCESS", "records": count})
    except Exception as e:
        pipeline_status.append({"table": tbl_cfg["table_name"], "status": "FAILED", "error": str(e)})
        print(f"  ✗ {tbl_cfg['table_name']}: FAILED — {e}")

## 6A. Lakebase Partners — Full or Incremental Load (Exercise 5)

Steps 1 → 4 of the Bronze algorithm at the top of this notebook:
**6A.1** secure connection → **6A.2** watermark → **6A.3** read from Lakebase → **6A.4** load into Bronze → **6A.5** run it.

### 6A.1 Secure connection helper

In [ ]:
import time

import psycopg
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import EndpointType

# WorkspaceClient() automatically uses the identity running this notebook / job.
# No token or password is written anywhere in this notebook.
w = WorkspaceClient()


def find_read_write_endpoint():
    """Return the read-write compute endpoint of our Lakebase branch."""
    branch_path = f"projects/{LAKEBASE_PROJECT_ID}/branches/{LAKEBASE_BRANCH_ID}"
    for endpoint in w.postgres.list_endpoints(parent=branch_path):
        if endpoint.status.endpoint_type == EndpointType.ENDPOINT_TYPE_READ_WRITE:
            return endpoint
    raise RuntimeError(f"No read-write endpoint found on {branch_path}")


def get_lakebase_connection():
    """
    Open a secure connection to Lakebase:
      1. find the database host (the read-write endpoint)
      2. ask Databricks for a short-lived OAuth token (valid 1 hour) and use it as the password
      3. connect with SSL; retry a few times because an idle database may need a moment to wake up
    """
    endpoint = find_read_write_endpoint()
    user_name = w.current_user.me().user_name

    for attempt in range(1, 4):
        try:
            credential = w.postgres.generate_database_credential(endpoint=endpoint.name)
            return psycopg.connect(
                host=endpoint.status.hosts.host,
                port=5432,
                dbname=LAKEBASE_DATABASE,
                user=user_name,
                password=credential.token,  # short-lived OAuth token, never printed or stored
                sslmode="require",          # encrypted connection only
                autocommit=True,
            )
        except psycopg.OperationalError as error:
            print(f"  … Lakebase not ready yet (attempt {attempt}/3): {error}")
            time.sleep(attempt * 5)
    raise RuntimeError("Could not connect to Lakebase after 3 attempts")

### 6A.2 The watermark — read it and save it

In [ ]:
def read_watermark():
    """The newest updated_at loaded so far — or None when nothing has been loaded yet."""
    row = spark.table(WATERMARK_TABLE).first()
    return None if row is None else row["last_updated_at"]


def save_watermark(new_watermark, rows_loaded):
    """Replace the single watermark row with the new value."""
    spark.sql(f"""
        INSERT OVERWRITE {WATERMARK_TABLE}
        SELECT '{SOURCE_TABLE}'        AS source_table,
               '{new_watermark}'       AS last_updated_at,
               '{LAKEBASE_LOAD_TYPE}'  AS last_load_type,
               {rows_loaded}           AS rows_loaded,
               current_timestamp()     AS updated_at
    """)

### 6A.3 Read partners from Lakebase — all of them, or only the changed ones

In [ ]:
PARTNER_COLUMNS = """
    partner_id, partner_name, phone, email, city, vehicle_type, partner_status,
    shift_preference, rating, joined_date, kyc_verified, created_at, updated_at
"""
READ_ALL_SQL = f"SELECT {PARTNER_COLUMNS} FROM {SOURCE_TABLE}"
READ_CHANGED_SQL = f"SELECT {PARTNER_COLUMNS} FROM {SOURCE_TABLE} WHERE updated_at > %s::timestamptz"


def read_partners_from_lakebase(watermark):
    """watermark = None → every partner.  Otherwise → only partners changed after the watermark."""
    with get_lakebase_connection() as conn:
        if watermark is None:
            return conn.execute(READ_ALL_SQL).fetchall()
        return conn.execute(READ_CHANGED_SQL, [watermark]).fetchall()

### 6A.4 Load into Bronze

In [ ]:
# The same columns as PARTNER_COLUMNS, with their Spark types
LAKEBASE_SCHEMA = """
    partner_id STRING, partner_name STRING, phone STRING, email STRING, city STRING,
    vehicle_type STRING, partner_status STRING, shift_preference STRING, rating DECIMAL(3,2),
    joined_date DATE, kyc_verified BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP
"""


def ingest_lakebase_partners():
    """Bronze steps 1 → 4. Returns the number of rows loaded."""
    # Steps 1 + 2 — what do we read?
    if LAKEBASE_LOAD_TYPE == "full":
        watermark = None                # full load: ignore the watermark, read every partner
    else:
        watermark = read_watermark()    # incremental: None on the very first run → read every partner

    if watermark is None:
        print(f"  {LAKEBASE_LOAD_TYPE.upper()} load: reading every partner")
    else:
        print(f"  {LAKEBASE_LOAD_TYPE.upper()} load: reading partners changed after {watermark}")
    rows = read_partners_from_lakebase(watermark)
    print(f"  {len(rows):,} rows read from Lakebase")

    if not rows:
        print("  Nothing changed since the last run — Bronze and the watermark stay as they are")
        return 0

    # Step 3 — write to Bronze: full → overwrite, incremental → append
    write_mode = "overwrite" if LAKEBASE_LOAD_TYPE == "full" else "append"
    bronze_df = (
        spark.createDataFrame(rows, LAKEBASE_SCHEMA)
        .withColumn("_load_type", lit(LAKEBASE_LOAD_TYPE))
        .withColumn("_source_system", lit("lakebase"))
        .withColumn("_bronze_loaded_at", current_timestamp())
    )
    bronze_df.write.format("delta").mode(write_mode).saveAsTable(BRONZE_PARTNERS_TABLE)

    # Step 4 — the new watermark = newest updated_at among the rows we just read (updated_at is the last column)
    new_watermark = max(row[-1] for row in rows).isoformat()
    save_watermark(new_watermark, len(rows))
    print(f"  ✓ {BRONZE_PARTNERS_TABLE}: {len(rows):,} rows ({write_mode}) — new watermark = {new_watermark}")
    return len(rows)

### 6A.5 Run it

In [ ]:
print(f"\n{'='*50}")
print(f"Lakebase → {BRONZE_PARTNERS_TABLE}")
print(f"{'='*50}")
try:
    lakebase_rows = ingest_lakebase_partners()
    total_records += lakebase_rows
    pipeline_status.append({"table": "delivery_partners", "status": "SUCCESS", "records": lakebase_rows,
                            "load_type": LAKEBASE_LOAD_TYPE})
except Exception as e:
    pipeline_status.append({"table": "delivery_partners", "status": "FAILED", "error": str(e)})
    print(f"  ✗ delivery_partners: FAILED — {e}")

## 7. Pipeline Summary & Data Quality Checks

In [ ]:
import json
from datetime import datetime

summary = {
    "pipeline": "bronze_ingestion",
    "catalog": CATALOG,
    "schema": SCHEMA,
    "env": ENV,
    "file_sources_load": "full (always overwritten)",
    "lakebase_load_type": LAKEBASE_LOAD_TYPE,
    "run_timestamp": datetime.utcnow().isoformat(),
    "total_records_ingested": total_records,
    "table_results": pipeline_status,
}

print(json.dumps(summary, indent=2))

In [ ]:
# Basic row-count validation
for tbl_cfg in pipeline_tables:
    tbl = tbl_cfg["table_name"]
    fq_table = f"{FQN}.{tbl}"
    count = spark.table(fq_table).count()
    assert count > 0, f"Data quality check FAILED: {fq_table} has 0 records!"
    print(f"  ✓ {fq_table}: {count:,} rows — PASS")

# Lakebase partners (Exercise 5) — fail the task if the load failed or the Bronze table is empty
lakebase_status = [s for s in pipeline_status if s["table"] == "delivery_partners"][0]
assert lakebase_status["status"] == "SUCCESS", f"Lakebase load FAILED: {lakebase_status}"
partner_rows = spark.table(BRONZE_PARTNERS_TABLE).count()
assert partner_rows > 0, f"Data quality check FAILED: {BRONZE_PARTNERS_TABLE} has 0 records!"
print(f"  ✓ {BRONZE_PARTNERS_TABLE}: {partner_rows:,} rows — PASS (watermark: {read_watermark()})")

print(f"\n🟢 Bronze ingestion pipeline completed successfully.")
print(f"   All tables created under: {FQN}.*")

## 8. Verify Isolation — List Only Our Objects

In [ ]:
print(f"Tables in {FQN}:")
spark.sql(f"SHOW TABLES IN {FQN}").show(truncate=False)